# Objective 1: build the 10-K dataset (laptop steps)

Run the cells from top to bottom. Use the **.venv** kernel (top right).

This notebook covers the steps that run on the laptop. Each step reads files and writes files, so you can close the notebook between steps and pick up later.

## Step 1.1: download the 24 filings from SEC EDGAR

- **The 8 training companies:** their FY2024 and FY2025 10-Ks.
- **The 8 unseen companies** (CRWV, FIG, ORCL, AMD, FFIV, AKAM, SWKS, JKHY): their FY2025 10-K only. FFIV, AKAM, SWKS and JKHY were added on Nov 26.
- **The fiscal year** is the calendar year in which the filing's period ends, so NVIDIA's FY2025 is the year ending 2025-01-26. 10-K/A amendments are skipped.
- **Each filing** goes to `data/raw/<TICKER>_FY<year>/`, together with a `metadata.json` file.
- **`data/filings.csv`** lists all 24 filings with their accession numbers (the SEC's ID for each filing), so anyone can re-download the same ones.

In [ ]:
# Setup: find the project folder, load config.yaml, and tell EDGAR who you are.
import csv
import json
import os
import time
from pathlib import Path

import yaml
from edgar import Company, set_identity

# The notebook may run from the notebooks/ folder, so walk up until config.yaml is found.
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
MANIFEST = ROOT / "data" / "filings.csv"

config = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))

identity = os.environ.get("SEC_IDENTITY")
assert identity, "Set the SEC_IDENTITY environment variable, then restart VS Code."
set_identity(identity)  # the SEC requires a name and email with every request

print("Project folder:", ROOT)

In [ ]:
# The list of (ticker, fiscal year) pairs to download.
unseen = config["heldout_companies"] + config["test_only_companies"]
training = [c for c in config["companies"] if c not in unseen]

pairs = [(t, fy) for t in training for fy in config["fiscal_years"]]  # 8 companies x 2 years
pairs += [(t, config["test_fy"]) for t in unseen]                   # 8 companies x FY2025

print(len(pairs), "filings:", pairs)

In [ ]:
# Two functions: find the right 10-K, and save it.

def find_10k(ticker, fiscal_year):
    """The single original 10-K whose period ends in `fiscal_year`."""
    filings = Company(ticker).get_filings(form="10-K", amendments=False)
    matches = [f for f in filings
               if f.form == "10-K" and str(f.period_of_report)[:4] == str(fiscal_year)]
    if len(matches) != 1:
        periods = [str(f.period_of_report) for f in filings]
        raise RuntimeError(f"{ticker} FY{fiscal_year}: expected 1 10-K, found {len(matches)}; periods {periods}")
    return matches[0]


def download(ticker, fiscal_year, force=False):
    """Save the main 10-K document, any Exhibit 13, and metadata.json. Returns the metadata."""
    folder = RAW / f"{ticker}_FY{fiscal_year}"
    meta_path = folder / "metadata.json"
    if meta_path.exists() and not force:
        print(f"{ticker} FY{fiscal_year}: already downloaded")
        return json.loads(meta_path.read_text(encoding="utf-8"))

    filing = find_10k(ticker, fiscal_year)
    folder.mkdir(parents=True, exist_ok=True)

    # The main 10-K document
    main_name = filing.primary_documents[0].document
    (folder / main_name).write_text(filing.html(), encoding="utf-8")

    # Exhibit 13: only if the company put its financial statements there instead of Item 8
    exhibit_13 = [a for a in filing.exhibits if (a.document_type or "").upper().startswith("EX-13")]
    for a in exhibit_13:
        content = a.download()
        if isinstance(content, bytes):
            content = content.decode("utf-8", errors="replace")
        (folder / a.document).write_text(content, encoding="utf-8")

    meta = {
        "ticker": ticker,
        "fiscal_year": fiscal_year,
        "form": filing.form,
        "company": filing.company,
        "cik": filing.cik,
        "period_of_report": str(filing.period_of_report),
        "filing_date": str(filing.filing_date),
        "accession_number": filing.accession_no,
        "url": filing.homepage_url,
        "main_document": main_name,
        "exhibit_13": [a.document for a in exhibit_13],
        "exhibits": [{"type": a.document_type or "", "document": a.document} for a in filing.exhibits],
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"{ticker} FY{fiscal_year}: saved {main_name} (period {meta['period_of_report']}, filed {meta['filing_date']})")
    return meta

In [ ]:
# Download all 24. Takes about a minute; filings already saved are skipped unless FORCE = True.
# FORCE = True downloads again even if a filing is already saved.
FORCE = False

rows, failures = [], []
for ticker, fy in pairs:
    try:
        rows.append(download(ticker, fy, force=FORCE))
    except Exception as error:  # keep going, and report at the end
        failures.append(f"{ticker} FY{fy}: {error}")
        print(f"{ticker} FY{fy}: FAILED: {error}")
    time.sleep(0.5)  # stay well under the SEC's limit of 10 requests per second

print(f"\n{len(rows)} downloaded, {len(failures)} failed")
for f in failures:
    print("  ", f)

In [ ]:
# Write data/filings.csv: the list of filings, which goes into git.
fields = ["ticker", "fiscal_year", "company", "period_of_report", "filing_date",
          "accession_number", "main_document", "exhibit_13", "url"]
with MANIFEST.open("w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        writer.writerow({**row, "exhibit_13": ";".join(row["exhibit_13"])})

print("Wrote", MANIFEST.relative_to(ROOT), "with", len(rows), "filings")

In [ ]:
# Check: is each file a complete 10-K? Looks for the main Items and any Exhibit 13.
import html
import re

for row in rows:
    folder = RAW / f"{row['ticker']}_FY{row['fiscal_year']}"
    raw_html = (folder / row["main_document"]).read_text(encoding="utf-8")
    text = html.unescape(re.sub(r"<[^>]+>", " ", raw_html))  # strip the HTML tags
    text = re.sub(r"\s+", " ", text)
    items = [i for i in ["1A", "1C", "7", "7A", "8", "9A"]
             if re.search(r"Item\s*" + i + r"\b", text, re.I)]
    print(f"{folder.name:13} {len(text.split()) / 1000:4.0f}k words   Items found: {', '.join(items):22} Exhibit 13: {row['exhibit_13'] or 'none'}")

## Step 1.2: split each filing into its Items

- **What an Item is:** a part of the 10-K that the SEC requires, such as Item 1A Risk Factors, Item 7 MD&A or Item 8 Financial Statements. Items are the top level of every table of contents we build.
- **How:** `edgartools` finds where each Item starts, using the filing's own table of contents, and gives each Item back as Markdown. In Markdown, `**text**` means bold, so headings show up as `**bold**` lines on their own.
- **Kept:** Items 1 to 14. Item 15 is dropped, because it's mainly the exhibit index (a list of attached documents), and so is any Item under 50 words, such as "Item 4. Mine Safety Disclosures: Not applicable."
- **The exception (Nov 26):** some companies, such as NVDA and ORCL, file their financial statements under Item 15, and their Item 8 is one sentence pointing there. When Item 8 has under 100 words, Item 15 is kept, under its own name, because that's where the statements are.
- **Saved to:** `data/parsed/<TICKER>_FY<year>/items.jsonl`, one line per Item. This folder isn't committed to git, because this notebook can rebuild it.
- **Headings set by size (Nov 26):** most filings put headings in bold, but FIG makes them bigger instead (24pt and 16pt, against 10pt body text). So before parsing, any text at 16pt or more is marked bold. This changes only FIG; the other 23 filings come out exactly the same.
- **Safety check:** step 1.3 finds the headings inside each Item from those bold lines. So for every Item we count the bold heading lines. A long Item with almost none means its headings were lost, for example because they're underlined rather than bold, and that filing then gets split by hand from its HTML instead.


In [ ]:
# Two functions: split one filing into its Items, and count heading lines.
import re

import pandas as pd
from edgar.documents import ParserConfig, parse_html

PARSED = ROOT / "data" / "parsed"
MIN_WORDS = 50          # Items shorter than this are dropped ("Not applicable")
DROP_ITEMS = {"15"}     # Item 15 is mainly the exhibit index...
POINTER_WORDS = 100     # ...unless Item 8 is shorter than this: then Item 8 only points to Item 15, which holds the statements

HEADING_PT = 16        # text at least this big (in points) counts as a heading, even if it isn't bold

# A heading line: a whole line in bold, e.g. "**Liquidity and Capital Resources**"
HEADING_LINE = re.compile(r"^\*\*[^*\n]{2,150}\*\*$")


def count_headings(markdown):
    lines = [line.strip() for line in markdown.splitlines() if line.strip()]
    return sum(1 for line in lines[1:] if HEADING_LINE.match(line))  # skip the Item's own title


def bold_large_text(html_text):
    """Mark text at HEADING_PT or bigger as bold, so headings set by size come out as **bold** lines."""
    def fix(match):
        style = match.group(1)
        size = re.search(r"font-size:\s*([\d.]+)pt", style)
        if size and float(size.group(1)) >= HEADING_PT:
            if "font-weight" in style:
                style = re.sub(r"font-weight:\s*\d+", "font-weight:700", style)
            else:
                style += ";font-weight:700"
        return f'style="{style}"'
    return re.sub(r'style="([^"]*)"', fix, html_text)


def split_items(folder):
    """Split one saved filing into Items. Returns a list of dicts, one per kept Item."""
    meta = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
    html_text = (folder / meta["main_document"]).read_text(encoding="utf-8")
    doc = parse_html(bold_large_text(html_text), ParserConfig(form="10-K"))

    sections = []
    for key, section in doc.sections.items():           # keys look like "part_ii_item_7a"
        item = key.split("_item_")[-1].upper()             # -> "7A"
        markdown = section.markdown() if callable(section.markdown) else section.markdown
        sections.append((key, item, section, markdown, len(markdown.split())))

    # If Item 8 is only a pointer (NVDA, ORCL), the financial statements are in Item 15, so keep it.
    item8_words = sum(words for _, item, _, _, words in sections if item == "8")
    drop = DROP_ITEMS - {"15"} if item8_words < POINTER_WORDS else DROP_ITEMS

    items = []
    for key, item, section, markdown, words in sections:
        if item in drop or words < MIN_WORDS:
            continue
        items.append({
            "item": item,
            "part": key.split("_item_")[0].replace("part_", "").upper(),   # "II"
            "title": section.title,
            "words": words,
            "bold_headings": count_headings(markdown),
            "confidence": section.confidence,
            "method": section.detection_method,
            "markdown": markdown,
        })
    return items

In [ ]:
# Split all 24 filings and save each one's Items. Takes a few minutes.
summary = []
for folder in sorted(RAW.glob("*_FY*")):
    items = split_items(folder)
    out_folder = PARSED / folder.name
    out_folder.mkdir(parents=True, exist_ok=True)
    with (out_folder / "items.jsonl").open("w", encoding="utf-8") as out:
        for it in items:
            out.write(json.dumps(it) + "\n")
    for it in items:
        summary.append({"filing": folder.name, **{k: v for k, v in it.items() if k != "markdown"}})
    kept_15 = "  (Item 15 kept: it holds the financial statements)" if any(it["item"] == "15" for it in items) else ""
    print(f"{folder.name:13} {len(items):2} Items kept{kept_15}")

summary = pd.DataFrame(summary)

In [ ]:
# Check 1: words in each Item, one row per filing. A blank means the Item wasn't found or was dropped.
order = ["1", "1A", "1B", "1C", "2", "3", "4", "5", "6", "7", "7A", "8", "9", "9A", "9B", "9C", "10", "11", "12", "13", "14", "15"]
words = summary.pivot(index="filing", columns="item", values="words")
words = words[[c for c in order if c in words.columns]]
words.fillna("").astype(str).replace(r"\.0$", "", regex=True)

In [ ]:
# Check 2: the safety check and other warnings.
CORE = ["1", "1A", "7", "8"]
problems = []
for filing, group in summary.groupby("filing"):
    found = set(group["item"])
    if "15" in found:
        found.add("8")             # the financial statements are in Item 15 (see step 1.2's notes)
    for item in CORE:
        if item not in found:
            problems.append((filing, item, "core Item missing"))
    for _, row in group.iterrows():
        if row["words"] >= 3000 and row["bold_headings"] < 3:
            problems.append((filing, row["item"], f"{row['words']} words but only {row['bold_headings']} bold headings"))
        if row["confidence"] < 0.8:
            problems.append((filing, row["item"], f"low confidence {row['confidence']:.2f} ({row['method']})"))
        # Raised from 40,000 on Nov 26: CRWV 1A (40,324), FIG 1A (39,158) and DELL 8 (45,987) were
        # checked by hand and are real long sections, each ending where it should.
        if row["words"] > 50000:
            problems.append((filing, row["item"], f"suspiciously long: {row['words']} words"))

print(len(problems), "problems")
pd.DataFrame(problems, columns=["filing", "item", "problem"])

In [ ]:
# Check 3: bold headings found in each core Item (step 1.3 builds the table of contents from these).
heads = summary[summary["item"].isin(CORE)].pivot(index="filing", columns="item", values="bold_headings")
heads[[c for c in CORE if c in heads.columns]]

## Step 1.3: find the headings inside each Item

Each Item is split into sections at its headings. These sections become the entries of the table of contents that STAIR chooses from, so they matter more than anything else in objective 1. The five rules agreed on Nov 26:

1. **What counts as a heading:** a whole line in bold, of 12 words or fewer. Longer bold lines, and bold sentences ending in a full stop, stay as text inside their section. That mostly affects Item 1A: each risk starts with a bold sentence, so Item 1A's sections are the risk groups, such as "Risks Related to Our Business". Many companies list their risk groups twice in Item 1A, first in a short "Risk Factor Summary" and then over the full risks; the summary's copy of each group heading stays as text inside the summary, so each group is one section. (FFIV's individual risks still come out as headings, because their italics are lost in the Markdown; they are removed by hand in step 1.6.)
2. **Noise is removed first:** "Table of Contents" page headers, page numbers, the company's name repeated at the top of each page, "(continued)" lines, the exhibit index and signatures sections (ORCL, FFIV), and any list of exhibits at the end of an Item (NVDA's Item 15, CRM's Item 16).
3. **Three levels at most:** Item → heading → subheading.
   - **Items 8 and 15 (the financial statements):** each statement and each Note is a heading, and the bold lines inside a Note are its subheadings. The Notes start after the line "Notes to Consolidated Financial Statements". A Note title looks like "Note 7 – Debt" or "7. Debt". JKHY's Note titles aren't bold, and ORCL's are split over two lines, so both cases are handled.
   - **Every other Item:** one level of headings. Bold lines there don't show which is a heading and which a subheading.
4. **Tiny sections are joined:** a section with under 30 words of its own text is joined with the next one (or the one before, if it's the last). Its heading stays in the text, in bold. Text before an Item's first heading goes to its first section, so only the lowest-level sections hold text.
5. **Saved to** `data/parsed/<TICKER>_FY<year>/outline.jsonl`, one line per entry, in the filing's order: `item`, `level` (1 = Item, 2 = heading, 3 = subheading), `title`, `is_leaf` (true if it has no sections under it), `words` and `text` (empty except for leaves). Step 1.4 numbers the entries and writes `toc.json`.

Items take the SEC's standard names (e.g. Item 7 is "Management's Discussion and Analysis of Financial Condition and Results of Operations"), so every filing names its Items the same way.

In [ ]:
# The step 1.3 functions: find the headings in each Item and build the filing's outline.
import json
import random
import re
from collections import Counter

import pandas as pd

PARSED = ROOT / "data" / "parsed"

MAX_HEADING_WORDS = 12   # rule 1: a heading is a whole bold line of at most this many words
MIN_SECTION_WORDS = 30   # rule 4: a section with less text than this is joined with the next one
STANDARD_ITEM = re.compile(r"^\d{1,2}[A-C]?$")   # keeps "7A", drops "EXHIBIT_INDEX" and "PART_IV_SIGNATURES"

ITEM_TITLES = {
    "1": "Business", "1A": "Risk Factors", "1B": "Unresolved Staff Comments", "1C": "Cybersecurity",
    "2": "Properties", "3": "Legal Proceedings", "4": "Mine Safety Disclosures",
    "5": "Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities",
    "6": "[Reserved]", "7": "Management's Discussion and Analysis of Financial Condition and Results of Operations",
    "7A": "Quantitative and Qualitative Disclosures About Market Risk",
    "8": "Financial Statements and Supplementary Data",
    "9": "Changes in and Disagreements with Accountants on Accounting and Financial Disclosure",
    "9A": "Controls and Procedures", "9B": "Other Information",
    "9C": "Disclosure Regarding Foreign Jurisdictions that Prevent Inspections",
    "10": "Directors, Executive Officers and Corporate Governance", "11": "Executive Compensation",
    "12": "Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters",
    "13": "Certain Relationships and Related Transactions, and Director Independence",
    "14": "Principal Accountant Fees and Services", "15": "Exhibits and Financial Statement Schedules",
    "16": "Form 10-K Summary",
}

BOLD_LINE = re.compile(r"^(?:\*\*[^*\n]+\*\*\s*)+$")            # a line made only of bold pieces
NUMBER_ONLY = re.compile(r"^(note\s+)?\d{1,2}\.?$", re.I)        # ORCL puts "1." on its own line
NOTE_TITLE = re.compile(r"^(note\s+\d{1,2}\b|\d{1,2}\s*[.:\-–—]\s*\S)", re.I)   # "Note 7 – Debt", "7. Debt"
PLAIN_NOTE = re.compile(r"^note\s+\d{1,2}\b", re.I)               # JKHY's note titles aren't bold
SCHEDULE = re.compile(r"^schedule\s+[ivx]+\b", re.I)
NOTES_START = re.compile(r"^notes to (the )?(consolidated )?financial statements", re.I)
NOT_A_TITLE = re.compile(r"^(\(|as of\b|for the (fiscal )?(years?|quarters?|periods?|three|six|nine|twelve)\b|"
                         r"(fiscal )?years? ended\b|in (millions|thousands)\b)", re.I)
SHORT_FORM_END = re.compile(r"\b(Inc|Corp|Co|Ltd|L\.P|N\.A|U\.S)\.$")   # "Adobe, Inc." is not a sentence
NAME_EXTRAS = {"inc", "corp", "corporation", "co", "company", "ltd", "plc", "and", "subsidiaries", "the"}


def tidy(text):
    """Remove Markdown escapes, bold marks and underline tags, and squeeze the spaces."""
    text = re.sub(r"\\(.)", r"\1", text)
    text = text.replace("**", " ").replace("<u>", "").replace("</u>", "")
    return re.sub(r"\s+", " ", text).strip()


def name_words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in NAME_EXTRAS]


def is_noise(line, company_words):
    """Rule 2: page headers and other lines that repeat on every page."""
    text = tidy(line)
    letters = re.sub(r"[^a-z]", "", text.lower())
    if letters in ("tableofcontents", "page"):                  # also catches "Table of Conten t s"
        return True
    if re.fullmatch(r"(page\s*)?(F\s*[-–—]\s*)?\d{1,3}|_+", text, re.I):
        return True                                               # page numbers like "57" or "F-12", and ____ lines
    if re.search(r"\bcontinued\b", text, re.I) and len(text.split()) <= MAX_HEADING_WORDS:
        return True                                               # "(continued)", ", continued"
    if NOTES_START.match(text) and len(text.split()) <= 8:
        return True                                               # "Notes to Consolidated Financial Statements"
    return len(text.split()) <= 8 and name_words(text) == company_words   # the company's name as a page header


def heading_title(line, in_financials):
    """The heading's title if this line is a heading, else None."""
    stripped = line.strip()
    if BOLD_LINE.match(stripped):
        return tidy(stripped)
    if in_financials and PLAIN_NOTE.match(tidy(stripped)) and len(stripped.split()) <= 15 and not stripped.endswith("."):
        return tidy(stripped)                                     # JKHY: "NOTE 7. DEBT"
    return None


def build_sections(row, company_words):
    """Split one Item into sections. Returns a list of dicts: level, title, text, is_note."""
    in_financials = row["item"] in ("8", "15")
    lines = row["markdown"].splitlines()[1:]                      # line 0 is the Item's own title

    # Pass 1: find the headings, joining a heading split over two lines (ORCL: "1." + "ORGANIZATION ...").
    parts = []                                                    # ("heading", title) or ("text", line)
    for line in lines:
        if not line.strip():
            continue
        if in_financials and NOTES_START.match(tidy(line)) and len(tidy(line).split()) <= 8:
            parts.append(("notes_start", None))                   # the notes begin here
            continue
        if is_noise(line, company_words):
            continue
        title = heading_title(line, in_financials)
        if title is not None:
            prev = parts[-1] if parts else None
            if not parts and title[:1].islower():
                continue                                          # the rest of a split Item title (ORCL: "Pro" + "perties")
            if prev and prev[0] == "heading" and (NUMBER_ONLY.match(prev[1]) or title[:1].islower()):
                parts[-1] = ("heading", f"{prev[1]} {title}")
                continue
            parts.append(("heading", title))
        else:
            parts.append(("text", line))

    # Cut any list of exhibits at the end of an Item (NVDA's Item 15, CRM's Item 16).
    for i, (kind, value) in enumerate(parts):
        if value is None:
            continue
        flat = re.sub(r"[^a-z]", "", value.lower())
        if (kind == "heading" and flat in ("exhibitindex", "indextoexhibits", "signatures")) or (
                kind == "text" and value.lstrip().startswith("|") and re.search(r"Exhibit (No|Number|Description)", value)):
            parts = parts[:i]
            break

    # Item 1A: the "Risk Factor Summary" repeats the risk-group headings. A heading that appears again
    # later in Item 1A is the summary's copy, so it stays as text and each group is one section.
    # Titles are compared by letters and digits only, so a comma doesn't matter (NVDA FY2025).
    same = lambda t: re.sub(r"[^a-z0-9]", "", t.lower())
    still_to_come = Counter(same(value.rstrip(":").strip()) for kind, value in parts if kind == "heading")

    # Pass 2: turn headings into sections, with levels.
    sections = [{"level": 2, "title": None, "text": [], "is_note": False}]   # text before the first heading
    notes_started = False
    has_marker = any(kind == "notes_start" for kind, _ in parts)
    for kind, value in parts:
        if kind == "notes_start":
            notes_started = True
            continue
        if kind == "text":
            sections[-1]["text"].append(value)
            continue
        title = value.rstrip(":").strip()
        still_to_come[same(title)] -= 1
        if row["item"] == "1A" and still_to_come[same(title)] > 0:
            sections[-1]["text"].append(f"**{title}**")         # the summary's copy of a risk-group heading
            continue
        too_long = len(title.split()) > MAX_HEADING_WORDS
        sentence = title.endswith(".") and len(title.split()) >= 5 and not SHORT_FORM_END.search(title)
        too_long = too_long or sentence                            # a risk factor's sentence, not a heading
        if too_long or NOT_A_TITLE.match(title) or not re.search(r"[A-Za-z]{2}", title):
            sections[-1]["text"].append(f"**{title}**")             # stays as text (e.g. a risk factor's sentence)
            continue
        if sections[-1]["title"] and sections[-1]["title"].lower() == title.lower():
            continue                                              # the same heading repeated on the next page
        is_note = in_financials and bool(NOTE_TITLE.match(title))
        if is_note and not notes_started:
            if has_marker:
                is_note = False                                   # a numbered line before the notes begin
            else:
                notes_started = True                              # no "Notes to ..." line: start at the first note
        level = 3 if (in_financials and notes_started and not is_note and any(s["is_note"] for s in sections)) else 2
        if SCHEDULE.match(title):
            level = 2                                             # "Schedule II – Valuation ..." comes after the notes
        sections.append({"level": level, "title": title, "text": [], "is_note": is_note})
    for s in sections:
        s["text"] = "\n\n".join(s["text"]).strip()
    return sections


def words(text):
    return len(text.split())


def to_tree(row, sections):
    """Arrange the flat sections under their Item: Item (level 1) -> heading (2) -> subheading (3)."""
    own_title = re.sub(r"^item\s+\w+\s*[.:\-–—]?\s*", "", tidy(row["markdown"].splitlines()[0]), flags=re.I)
    title = ITEM_TITLES.get(row["item"], own_title)                # the SEC's name; CRM's Item 4A keeps its own
    item = {"level": 1, "title": title, "text": sections[0]["text"], "children": []}
    for s in sections[1:]:
        node = {"level": s["level"], "title": s["title"], "text": s["text"], "children": []}
        if s["level"] == 3 and item["children"]:
            item["children"][-1]["children"].append(node)
        else:
            node["level"] = 2
            item["children"].append(node)
    return item


def first_leaf(node):
    return first_leaf(node["children"][0]) if node["children"] else node


def last_leaf(node):
    return last_leaf(node["children"][-1]) if node["children"] else node


def join_tiny(node):
    """Rule 4: a section under MIN_SECTION_WORDS is joined with the next one (or the one before, if it's last)."""
    for child in node["children"]:
        join_tiny(child)
    kept = []
    carry = ""
    for child in node["children"]:
        if carry:
            leaf = first_leaf(child)
            leaf["text"] = (carry + "\n\n" + leaf["text"]).strip()
            carry = ""
        if not child["children"] and words(child["text"]) < MIN_SECTION_WORDS:
            carry = f"**{child['title']}**\n\n{child['text']}".strip()
            continue
        kept.append(child)
    if carry:
        if kept:
            leaf = last_leaf(kept[-1])
            leaf["text"] = (leaf["text"] + "\n\n" + carry).strip()
        else:
            node["text"] = (node["text"] + "\n\n" + carry).strip()
    node["children"] = kept
    # A parent's own text (before its first heading) goes to its first section, so only leaves hold text.
    if node["children"] and node["text"]:
        leaf = first_leaf(node)
        leaf["text"] = (node["text"] + "\n\n" + leaf["text"]).strip()
        node["text"] = ""


def flatten(node, item):
    rows = [{"item": item, "level": node["level"], "title": node["title"], "is_leaf": not node["children"],
             "words": words(node["text"]), "text": node["text"]}]
    for child in node["children"]:
        rows += flatten(child, item)
    return rows


def outline_filing(folder, company):
    """Build the outline of one filing: one row per node, in document order."""
    company_words = name_words(company)
    rows = [json.loads(line) for line in (folder / "items.jsonl").open(encoding="utf-8")]
    outline = []
    for row in rows:
        if not STANDARD_ITEM.match(row["item"]):
            continue                                              # exhibit index, signatures
        tree = to_tree(row, build_sections(row, company_words))
        join_tiny(tree)
        if not tree["children"] and words(tree["text"]) < MIN_SECTION_WORDS:
            continue                                              # nothing left once noise is removed (CRM's Item 16)
        outline += flatten(tree, row["item"])
    return outline

In [ ]:
# Build and save the outline of all 24 filings. Takes about a minute.
outlines = {}
for folder in sorted(PARSED.glob("*_FY*")):
    company = json.loads((RAW / folder.name / "metadata.json").read_text(encoding="utf-8"))["company"]
    outline = outline_filing(folder, company)
    with (folder / "outline.jsonl").open("w", encoding="utf-8") as out:
        for entry in outline:
            out.write(json.dumps(entry) + "\n")
    outlines[folder.name] = outline
    print(f"{folder.name:13} {sum(e['is_leaf'] for e in outline):4} sections")

In [ ]:
# Check 1: one row per filing.
# "toc_tokens" is a rough guess at the table of contents' length in tokens (about 3.5 characters per token,
# plus the section numbers). The official code allows 2,048 tokens for the table of contents plus the question,
# so this shows how far that limit must rise; it is measured exactly with Mistral's tokenizer in objective 2.
rows = []
for filing, outline in outlines.items():
    leaves = [e for e in outline if e["is_leaf"]]
    longest = max(leaves, key=lambda e: e["words"])
    rows.append({
        "filing": filing,
        "items": sum(e["level"] == 1 for e in outline),
        "headings": sum(e["level"] == 2 for e in outline),
        "subheadings": sum(e["level"] == 3 for e in outline),
        "sections": len(leaves),
        "notes": sum(e["level"] == 2 and e["item"] in ("8", "15") and bool(NOTE_TITLE.match(e["title"])) for e in outline),
        "longest_section": f"{longest['words']} words (Item {longest['item']}: {longest['title'][:40]})",
        "toc_tokens": round(sum(len(e["title"]) / 3.5 + 4 for e in outline)),
    })
check = pd.DataFrame(rows).set_index("filing")
check

In [ ]:
# Check 2: warnings to look at by hand.
LONG_SECTION = 15000   # a section this long may hide headings that weren't found
warnings = []
for filing, outline in outlines.items():
    if check.loc[filing, "notes"] < 5:
        warnings.append((filing, "8/15", f"only {check.loc[filing, 'notes']} Notes found"))
    for e in outline:
        if e["level"] == 1 and e["is_leaf"] and e["words"] >= 3000:
            warnings.append((filing, e["item"], f"{e['words']} words but no headings found"))
        if e["is_leaf"] and e["words"] > LONG_SECTION:
            warnings.append((filing, e["item"], f"very long section: {e['title'][:50]} ({e['words']} words)"))
print(len(warnings), "warnings")
pd.DataFrame(warnings, columns=["filing", "item", "warning"])

In [ ]:
# Check 3: 20 headings picked at random from each filing, to read through for anything that isn't a real heading.
rng = random.Random(config["seed"])
for filing, outline in outlines.items():
    picked = rng.sample([e for e in outline if e["level"] > 1], 20)
    print(filing)
    print("   " + " | ".join(f"{e['item']}: {e['title'][:45]}" for e in picked) + "\n")

In [ ]:
# Check 4: one filing's whole outline, indented by level. Change SHOW to look at another filing.
SHOW = "FIG_FY2025"
for e in outlines[SHOW]:
    indent = "    " * (e["level"] - 1)
    label = f"Item {e['item']}. {e['title']}" if e["level"] == 1 else e["title"]
    print(f"{indent}{label}" + (f"  ({e['words']} words)" if e["is_leaf"] else ""))

## Steps 1.4 and 1.5: number the sections and write the table of contents and section text

Each outline from step 1.3 becomes the table of contents that STAIR reads. The model answers with a number and a title, such as "7.3 Liquidity and Capital Resources". The five rules agreed on Nov 26:

1. **Numbered by position.** The Item's own number comes first: Item 7's third heading is "7.3", its first subheading "7.3.1", and Item 1A's second heading "1A.2". An Item with no headings is just its number, e.g. "2 Properties". Numbers make every entry unique, even when titles repeat (GOOGL's MD&A has two "Google Cloud" sections).
2. **Titles as the filing writes them,** capitals and the filing's own note numbers included. So "8.17 Note 9 – Debt" means: the 17th heading in Item 8, which the filing calls Note 9.
3. **Every entry is listed, but only sections with their own text are answers.** Items and Notes that only group other sections stay in the table of contents to show its structure.
4. **Three files per filing,** in `data/parsed/<TICKER>_FY<year>/`:
   - `toc.json`: the official code's format, `{"title": "SEC Form 10-K filing", "table_of_contents": [{"id", "section_num", "title"}]}`. The title has no company name or year.
   - `nodes.jsonl`: one line per entry with `id`, `section_num`, `title`, `level`, `parent` (the number of the entry it sits under) and `is_leaf` (true if it holds text and can be an answer).
5. **`content.jsonl` (step 1.5):** one line per answerable section, with `id`, `title` and `text`. The text is plain: tables become rows with cells separated by " | ", and "$" and "%" are attached to their numbers.

**Check:** each table of contents is counted in real tokens with Mistral's own tokenizer, the model STAIR is trained from. The official code allows 2,048 tokens for the table of contents, the question and the answer together, so this shows how far that limit has to rise in objective 2. The tokenizer (a few MB) downloads from Hugging Face the first time, using `HF_TOKEN`. A warning about "symlinks" on Windows is harmless: it only means the download cache uses a little more disk space.

In [ ]:
# The step 1.4 functions: number each filing's sections and write toc.json, nodes.jsonl and content.jsonl.
import json
import re

import pandas as pd

PARSED = ROOT / "data" / "parsed"

TOC_TITLE = config["prompt"]["header"]            # "SEC Form 10-K filing": no company name or year
TABLE_DIVIDER = re.compile(r"^\|[\s\-:|]+\|$")   # Markdown's "| --- | --- |" line under a table's first row


def table_row(line):
    """One table row as "cell | cell | cell", with "$" and "%" attached to their numbers."""
    cells = [c.strip() for c in line.strip("|").split("|")]
    cells = [c for c in cells if c]                                # drop the empty cells
    joined = []
    for cell in cells:
        if joined and joined[-1] == "$":
            joined[-1] = "$" + cell                                # "$ | 8,589" -> "$8,589"
        elif joined and cell in ("%", ")", "%)"):
            joined[-1] += cell                                     # "12 | %" -> "12%"
        else:
            joined.append(cell)
    return " | ".join(joined)


def plain_text(markdown):
    """Rule 5: Markdown to plain text. Paragraphs are separated by a blank line; a table's rows are kept together."""
    out, prev_was_row = [], False
    for line in markdown.splitlines():
        line = line.strip()
        if not line or TABLE_DIVIDER.match(line):
            continue
        line = re.sub(r"\\(.)", r"\1", line)                       # Markdown escapes: "1\." -> "1."
        line = line.replace("**", "").replace("<u>", "").replace("</u>", "")
        is_row = line.startswith("|")
        if is_row:
            line = table_row(line)
        line = re.sub(r"\s+", " ", line).strip()
        if not line:
            continue
        if out:
            out.append("\n" if (is_row and prev_was_row) else "\n\n")
        out.append(line)
        prev_was_row = is_row
    return "".join(out)


def number_outline(outline):
    """Rule 1: number every entry by its position. Item 7's third heading is "7.3", its first subheading "7.3.1"."""
    nodes, counters = [], {}
    stack = []                                                     # section numbers of the current Item and heading
    for entry in outline:
        level = entry["level"]
        if level == 1:
            num = entry["item"]
        else:
            parent = stack[level - 2]
            counters[parent] = counters.get(parent, 0) + 1
            num = f"{parent}.{counters[parent]}"
        stack = stack[:level - 1] + [num]
        nodes.append({
            "id": num,
            "section_num": num,
            "title": re.sub(r"\s+", " ", entry["title"]).strip(),  # rule 2: as the filing writes it
            "level": level,
            "parent": stack[level - 2] if level > 1 else None,
            "is_leaf": entry["is_leaf"],
            "text": entry["text"],
        })
    return nodes


def write_filing(folder):
    """Write the three files for one filing. Returns its nodes."""
    outline = [json.loads(line) for line in (folder / "outline.jsonl").open(encoding="utf-8")]
    nodes = number_outline(outline)
    targets = [f"{n['section_num']} {n['title']}" for n in nodes]
    assert len(set(targets)) == len(targets), f"{folder.name}: two entries have the same number and title"

    # Rule 4: toc.json in the official code's format, and nodes.jsonl with the fuller details.
    toc = {"title": TOC_TITLE,
           "table_of_contents": [{"id": n["id"], "section_num": n["section_num"], "title": n["title"]} for n in nodes]}
    (folder / "toc.json").write_text(json.dumps(toc, indent=1, ensure_ascii=False), encoding="utf-8")
    with (folder / "nodes.jsonl").open("w", encoding="utf-8") as out:
        for n in nodes:
            out.write(json.dumps({k: n[k] for k in ("id", "section_num", "title", "level", "parent", "is_leaf")},
                                 ensure_ascii=False) + "\n")

    # Rule 5 (step 1.5): one line per answerable section, i.e. each section with its own text.
    with (folder / "content.jsonl").open("w", encoding="utf-8") as out:
        for n in nodes:
            if n["is_leaf"]:
                out.write(json.dumps({"id": n["id"], "title": n["title"], "text": plain_text(n["text"])},
                                     ensure_ascii=False) + "\n")
    return nodes


def toc_as_text(nodes):
    """The table of contents as the model reads it: one "number title" line per entry."""
    return "\n".join(f"{n['section_num']} {n['title']}" for n in nodes)

In [ ]:
# Number every filing's sections and write toc.json, nodes.jsonl and content.jsonl. Takes a few seconds.
all_nodes = {}
for folder in sorted(PARSED.glob("*_FY*")):
    all_nodes[folder.name] = write_filing(folder)
    nodes = all_nodes[folder.name]
    print(f"{folder.name:13} {len(nodes):4} entries, {sum(n['is_leaf'] for n in nodes):4} answerable sections")

In [ ]:
# Check 1: each table of contents in real Mistral tokens. Loading the tokenizer takes about 30 seconds.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config["models"]["stair_base"])
rows = []
for filing, nodes in all_nodes.items():
    rows.append({
        "filing": filing,
        "entries": len(nodes),
        "answerable": sum(n["is_leaf"] for n in nodes),
        "deepest": max(n["level"] for n in nodes),
        "toc_tokens": len(tokenizer(toc_as_text(nodes), add_special_tokens=False)["input_ids"]),
    })
tokens = pd.DataFrame(rows).set_index("filing")
print(f"Longest table of contents: {tokens['toc_tokens'].max()} tokens ({tokens['toc_tokens'].idxmax()}); "
      f"{(tokens['toc_tokens'] > 2048).sum()} of {len(tokens)} are over 2,048 on their own.")
tokens

In [ ]:
# Check 2: one filing's table of contents as the model reads it, and one of its sections. Change SHOW to look at another.
SHOW = "NVDA_FY2025"
nodes = all_nodes[SHOW]
print(toc_as_text(nodes)[:2500], "\n...\n")
content = [json.loads(line) for line in (PARSED / SHOW / "content.jsonl").open(encoding="utf-8")]
section = next(c for c in content if "Balance Sheet" in c["title"])
print(f"Section {section['id']} {section['title']}:\n")
print(section["text"][:1200])